In [15]:
import torch
import numpy as np
from tqdm import tqdm
from scipy.stats import spearmanr
from sklearn.metrics import average_precision_score
from collections import defaultdict 
from poincare import PoincareManifold          # tes fichiers locaux
from model import Distance_PE
from data import G_hpo

In [23]:
# Modifier à chaque fois :
checkpoint = torch.load('models/poincare_hpo_0.5_500_NNEG10_32.pt', map_location='cpu', weights_only=False)

objects = checkpoint['objects']
node2id = checkpoint['node2id']
losses = checkpoint['losses']
norm_history = checkpoint['norm_history']
edges = checkpoint['edges']
data = checkpoint['data']
hp = checkpoint['hyperparams']

In [24]:
manifold = PoincareManifold()
model = Distance_PE(n=len(objects), dim=hp['dim'],
                       manifold=manifold, sparse=True)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

Distance_PE(
  (embeddings): Embedding(19389, 2, sparse=True)
)

In [25]:
W = model.weight.detach().cpu().numpy()   # (N, dim)
norms = np.linalg.norm(W, axis=1)

print(f"Modèle chargé — {len(objects)} nœuds | dim={hp['dim']} | "
      f"{len(losses)} epochs")
print(f"Norme moy={norms.mean():.4f} | max={norms.max():.4f}")

i_min = np.argmin(norms).item()
print(f"Index de la plus petite norme : {i_min}")
print(f"Position du point : {W[i_min]}")


Modèle chargé — 19389 nœuds | dim=2 | 500 epochs
Norme moy=0.8804 | max=0.9877
Index de la plus petite norme : 18885
Position du point : [0.03388773 0.01003373]


In [26]:
degrees = np.array([len(data.pos_neighbors[i]) for i in range(len(objects))])
norms = model.weight.detach().norm(dim=-1).numpy()

# Corrélation degré/norme attendue : négative
rho, pval = spearmanr(degrees, norms)
print(f"Corrélation Spearman degré/norme : {rho:.3f} (p={pval:.2e})")

Corrélation Spearman degré/norme : 0.202 (p=7.49e-178)


In [27]:
pos_neighbors = defaultdict(set)
pos_parents = defaultdict(set)

for u, v in edges:
    pos_neighbors[int(u)].add(int(v))
    pos_parents[int(v)].add(int(u))
    

len(pos_parents[i_min])
len(pos_parents[0])

7

In [28]:
degrees = np.array([len(data.pos_neighbors[i]) for i in range(len(objects))])

# Top 10 plus proches du centre
center_ids = np.argsort(norms)[:10]
print("=== 10 nœuds les plus proches du CENTRE ===")
for i in center_ids:
    print(f"  {objects[i]:<30} norme={norms[i]:.4f}  degré={degrees[i]}")

# Top 10 plus proches du bord
border_ids = np.argsort(norms)[-10:]
print("\n=== 10 nœuds les plus proches du BORD ===")
for i in border_ids:
    print(f"{objects[i]:<30} norme={norms[i]:.4f}  degré={degrees[i]}")

=== 10 nœuds les plus proches du CENTRE ===
  HP:6000959                     norme=0.0353  degré=2
  HP:0002086                     norme=0.0369  degré=1
  HP:0000707                     norme=0.0371  degré=1
  HP:0000118                     norme=0.0380  degré=1
  HP:0000769                     norme=0.0389  degré=1
  HP:0410017                     norme=0.0389  degré=1
  HP:0000372                     norme=0.0419  degré=1
  HP:0007609                     norme=0.0527  degré=1
  HP:0000969                     norme=0.0528  degré=1
  HP:0025533                     norme=0.0540  degré=1

=== 10 nœuds les plus proches du BORD ===
HP:0009691                     norme=0.9875  degré=2
HP:0009507                     norme=0.9875  degré=3
HP:0010262                     norme=0.9875  degré=2
HP:0009529                     norme=0.9875  degré=3
HP:0009351                     norme=0.9875  degré=3
HP:0009208                     norme=0.9875  degré=3
HP:0009680                     norme=0.9876  

In [12]:
def evaluate(model, objects, edges, node2id):
    model.eval()
    W  = model.weight.detach()   # (N, dim)
 
    pos_neighbors = defaultdict(set)
    for u, v in edges:
        pos_neighbors[int(u)].add(int(v))

    ranksum, ap_scores = 0, 0
    nranks = 0
    iters = 0
    labels = np.empty(model.embeddings.weight.size(0))
 
    for u in tqdm(objects):
        labels.fill(0)
        u = int(node2id[u])
        neighbors = pos_neighbors.get(u, set())
        if not neighbors :
            continue
        u_exp = W[u].unsqueeze(0).expand(W.shape[0], -1)  # Coordonnées de u dans la boule de Poincaré
        dists = manifold.distance(u_exp, W).numpy()  # Distance de Poincaré de u aux autres noeuds
        dists[u] = 1e12
        #order = np.argsort(dists)  # Tri par distance décroissante p/r à u
        sorted_ind = np.argsort(dists)

        #ranks = int(np.where(order == v)[0][0]) + 1  # Rang du noeud v p/r à u dans l'embedding
        #ranks.append(rank)
        ranks, = np.where(np.isin(sorted_ind, list(neighbors)))
        ranks += 1
        N = ranks.shape[0]

        ranksum += ranks.sum() - (N * (N - 1) / 2)
        nranks += ranks.shape[0]
        labels[list(neighbors)] = 1
        ap_scores += average_precision_score(labels, -dists)
        iters += 1

        #pos  = pos_neighbors[u]  # Voisins de u dans la représentation initiale
        #hits, psum = 0, 0.0 
        #for k, idx in enumerate(order[1:], 1):  # On parcourt les noeuds du plus proche au plus éloigné
            #if idx in pos:
                #hits  += 1
                #psum  += hits / k
        #aps.append(psum / max(len(pos), 1))
 
    return float(ranksum), nranks, ap_scores, iters

In [29]:
objects_hpo = list(G_hpo.nodes())
node2id_hpo = {n: i for i, n in enumerate(objects_hpo)}
edges_hpo = np.array([(node2id_hpo[v], node2id_hpo[u]) for u, v in G_hpo.edges()],dtype=np.int64)
new_edges = [(v, u) for u, v in edges]
results = evaluate(model, objects, new_edges, node2id)

print("Erreur moyenne sur le rang : ", float(results[0]) / results[1])
print("Mean Average Precision :", float(results[2]) / results[3])

100%|██████████| 19389/19389 [02:06<00:00, 153.30it/s] 

Erreur moyenne sur le rang :  917.6114794948685
Mean Average Precision : 0.6019127697734824
